# Length Bucketing Analysis

Run two 300-second pretrain jobs with length bucketing on and off, then compare throughput.

In [5]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path


current = Path.cwd().resolve()
for candidate in [current, *current.parents]:
    if (candidate / "pyproject.toml").exists():
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find repo root containing pyproject.toml")

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

RUNS_ROOT = Path("artifacts/runs")
LB_ON_CONFIG = Path("configs/lb_on.yaml")
LB_OFF_CONFIG = Path("configs/lb_off.yaml")
REPO_ROOT_PREFIX = str(REPO_ROOT) + "/"


In [6]:
completed_on = !uv run python run_pretrain.py --config {LB_ON_CONFIG} --runs-root {RUNS_ROOT} --note length_bucketing_on
print("\n".join(line.replace(REPO_ROOT_PREFIX, "") for line in completed_on))

step=50 pretrain_loss=7.5088 step_time=0.1338s tokens_per_second=55148.8 device=mps
step=100 pretrain_loss=7.1919 step_time=0.4283s tokens_per_second=37322.1 device=mps
step=150 pretrain_loss=6.9326 step_time=0.3429s tokens_per_second=39046.8 device=mps
step=200 pretrain_loss=6.8658 step_time=0.3740s tokens_per_second=40000.1 device=mps
step=250 pretrain_loss=7.0216 step_time=0.4430s tokens_per_second=36802.9 device=mps
step=300 pretrain_loss=6.2584 step_time=0.4336s tokens_per_second=37683.9 device=mps
step=350 pretrain_loss=5.8226 step_time=0.4711s tokens_per_second=34508.5 device=mps
step=400 pretrain_loss=5.0353 step_time=0.3207s tokens_per_second=46778.4 device=mps
step=450 pretrain_loss=5.1356 step_time=0.4466s tokens_per_second=36617.5 device=mps
step=500 pretrain_loss=4.2988 step_time=0.3453s tokens_per_second=41994.5 device=mps
step=550 pretrain_loss=4.2774 step_time=0.3289s tokens_per_second=48203.1 device=mps
step=600 pretrain_loss=3.8366 step_time=0.4300s tokens_per_second=

In [7]:
completed_off = !uv run python run_pretrain.py --config {LB_OFF_CONFIG} --runs-root {RUNS_ROOT} --note length_bucketing_off
print("\n".join(line.replace(REPO_ROOT_PREFIX, "") for line in completed_off))

step=50 pretrain_loss=7.5987 step_time=0.4336s tokens_per_second=15929.7 device=mps
step=100 pretrain_loss=7.4611 step_time=0.4320s tokens_per_second=16416.2 device=mps
step=150 pretrain_loss=6.7328 step_time=0.4329s tokens_per_second=16688.9 device=mps
step=200 pretrain_loss=6.9750 step_time=0.4326s tokens_per_second=13356.8 device=mps
step=250 pretrain_loss=6.2222 step_time=0.4306s tokens_per_second=20009.9 device=mps
step=300 pretrain_loss=6.5051 step_time=0.4480s tokens_per_second=14899.6 device=mps
step=350 pretrain_loss=6.4865 step_time=0.4329s tokens_per_second=20004.7 device=mps
step=400 pretrain_loss=5.6928 step_time=0.4326s tokens_per_second=16028.7 device=mps
step=450 pretrain_loss=5.5492 step_time=0.4359s tokens_per_second=22496.1 device=mps
step=500 pretrain_loss=4.5042 step_time=0.4322s tokens_per_second=12559.1 device=mps
step=550 pretrain_loss=4.9670 step_time=0.4345s tokens_per_second=20127.5 device=mps
step=600 pretrain_loss=4.6220 step_time=0.4337s tokens_per_second=

In [8]:
run_dir_on = None
for line in completed_on:
    if line.startswith("run_dir="):
        run_dir_on = Path(line.split("=", 1)[1].strip())
        break
if run_dir_on is None:
    raise ValueError("Could not find run_dir for lb_on")

run_dir_off = None
for line in completed_off:
    if line.startswith("run_dir="):
        run_dir_off = Path(line.split("=", 1)[1].strip())
        break
if run_dir_off is None:
    raise ValueError("Could not find run_dir for lb_off")

summary_on = json.loads((run_dir_on / "summary.json").read_text(encoding="utf-8"))
summary_off = json.loads((run_dir_off / "summary.json").read_text(encoding="utf-8"))
run_dir_on_display = run_dir_on.relative_to(REPO_ROOT) if run_dir_on.is_absolute() else run_dir_on
run_dir_off_display = run_dir_off.relative_to(REPO_ROOT) if run_dir_off.is_absolute() else run_dir_off

toks_on = float(summary_on["result"]["average_tokens_per_second"])
toks_off = float(summary_off["result"]["average_tokens_per_second"])
step_time_on = float(summary_on["result"]["average_step_time_seconds"])
step_time_off = float(summary_off["result"]["average_step_time_seconds"])
step_on = int(summary_on["result"]["final_step"])
step_off = int(summary_off["result"]["final_step"])
speedup_ratio = toks_on / toks_off
speedup_percent = (speedup_ratio - 1.0) * 100.0

print("length bucketing benchmark")
print()
print(f"lb_on : toks/sec={toks_on:.1f} step_time={step_time_on:.4f}s final_step={step_on} run_dir={run_dir_on_display}")
print(f"lb_off: toks/sec={toks_off:.1f} step_time={step_time_off:.4f}s final_step={step_off} run_dir={run_dir_off_display}")
print()
print(f"speedup_ratio={speedup_ratio:.4f}")
print(f"speedup_percent={speedup_percent:.2f}%")


length bucketing benchmark

lb_on : toks/sec=39332.4 step_time=0.3692s final_step=791 run_dir=artifacts/runs/2026-04-17_16-07-16_810176
lb_off: toks/sec=18425.3 step_time=0.4343s final_step=678 run_dir=artifacts/runs/2026-04-17_16-12-21_478587

speedup_ratio=2.1347
speedup_percent=113.47%
